# "The agent starts like an agent and stops like a coin flip" — CPU reproduction

This notebook re-computes the tables and figures of the ARC Prize 2026 Paper Track entry
*The agent starts like an agent and stops like a coin flip: a computational-neuroscience dissection of why
humans solve ARC-AGI-3 and an LLM agent does not* from the public dataset
[tantan0327/arc3-paper-artefacts](https://www.kaggle.com/datasets/tantan0327/arc3-paper-artefacts) (CC0), on CPU, in
minutes. The full paper (PDF), every script and the public snapshot of the repository are at
[github.com/tantan0327/arc-agi-3-agent](https://github.com/tantan0327/arc-agi-3-agent) (MIT-0). The linked Kaggle
submission is [tantan0327/arc3-franzen-dprime](https://www.kaggle.com/code/tantan0327/arc3-franzen-dprime)
(submission ID 57160861).

What runs here, and the paper section it reproduces:

| part | paper | needs |
|---|---|---|
| 1. The leaderboard is a max-statistic | §2.3, Figure 1, Appendix C | two CSVs in the dataset |
| 2. The allocation simulator: validation, calibration, policy ceiling, token-cost projection | §7.2, Figure 8, §9 table, Appendix D | `data/traces.json`, `scripts/sched_sim.py` |
| 3. "Slow, not stuck": the 60-minute traces | §7.1, Figure 7 | `data/traces.json` |
| 4. The 2026-10-24 arms against the base band | §6, Figure 10 | `data/arms_2026-10-24/*/benchmark.json` |
| 5. Theory induction, goal census, planner, UNDO — on the real game engine | §5.1, §5.2, §5.4, §6 | the engine (`pip install arc-agi arcengine`), the competition's game files, the human replays in the dataset |
| 6. The recorded outputs, for comparison | Appendix B | `results/*` |

Nothing here needs a GPU or a model. Part 5 replays 6,512 human actions through the engine and takes a few minutes.


In [ ]:
import os, sys, json, glob, csv, re, time, importlib.util, subprocess, statistics as st
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
try:
    from IPython.display import display, Image
except Exception:  # running outside a notebook (local test)
    def display(*a, **k): pass
    class Image:
        def __init__(self, filename=None, **k): self.filename = filename

def _find_dataset():
    env = os.environ.get("ARC3_PAPER_INPUT")
    cands = [env] if env else []
    cands += ["/kaggle/input/datasets/tantan0327/arc3-paper-artefacts", "/kaggle/input/arc3-paper-artefacts"]
    for c in cands:
        if c and Path(c, "data", "traces.json").exists():
            return Path(c)
    hits = list(Path("/kaggle/input").glob("**/data/traces.json")) if Path("/kaggle/input").exists() else []
    return hits[0].parent.parent if hits else Path(cands[-1])
IN = _find_dataset()
COMP = Path(os.environ.get("ARC3_COMP_INPUT", "/kaggle/input/competitions/arc-prize-2026-arc-agi-3"))
if not (COMP / "environment_files").exists() and Path("/kaggle/input").exists():
    _c = [p for p in Path("/kaggle/input").glob("**/environment_files") if p.is_dir()]
    COMP = _c[0].parent if _c else COMP
OUT = Path(os.environ.get("ARC3_OUT", "/kaggle/working" if Path("/kaggle/working").exists() else "."))
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

def show(fig, name):
    p = OUT / name; fig.savefig(p, dpi=130, bbox_inches="tight"); plt.close(fig); display(Image(filename=str(p)))

print("dataset:", IN, "| exists:", IN.exists())
for sub in ("data", "scripts", "results", "patches"):
    files = sorted(str(p.relative_to(IN)) for p in (IN / sub).rglob("*") if p.is_file())
    print(f"  {sub}/: {len(files)} files; e.g. {files[:3]}")
print("competition game files:", (COMP / "environment_files").exists(), "->", COMP / "environment_files")


## 1. The leaderboard is a max-statistic (§2.3, Figure 1)

Each team displays its *best* draw. The dataset carries every scored daily submission of ours
(`data/submissions.csv`: date, draw number, configuration family, public score) and the public leaderboard
as downloaded on 2026-10-24. A fixed configuration drawn daily has a per-draw standard deviation of about 3
at the 27-point level; the band's mean is the mean of maxima, two points above any single draw.


In [ ]:
subs = list(csv.DictReader(open(IN / "data/submissions.csv")))
for r in subs: r["public_score"] = float(r["public_score"]); r["draw"] = int(r["draw"]) if r["draw"] else None
fam = {}
for r in subs:
    if r["public_score"] > 0:  # 0.00 = the scoring environment failed before the model server was up (§2.4); excluded from every family mean
        fam.setdefault(r["family"], []).append(r["public_score"])
print(f"{'family':<30} {'n':>3} {'mean':>6} {'sd':>5} {'max':>6}  draws")
for name, xs in sorted(fam.items(), key=lambda kv: -len(kv[1])):
    if len(xs) >= 2:
        print(f"{name:<30} {len(xs):>3} {st.mean(xs):6.2f} {st.pstdev(xs):5.2f} {max(xs):6.2f}  {sorted(xs)}")
dp_all = [x for f, xs in fam.items() if f.startswith("D-prime") for x in xs]
print(f"{'D-prime family, all startup variants':<30} {len(dp_all):>3} {st.mean(dp_all):6.2f} {st.pstdev(dp_all):5.2f} {max(dp_all):6.2f}  (the paper's per-draw mean)")
zeros = sum(1 for r in subs if r['public_score'] == 0 and r['date'] >= '2026-10-10')
print(f"zero-score reruns since 2026-10-10: {zeros} (platform startup failures, see §2.4)")

lb = list(csv.DictReader(open(IN / "data/leaderboard_public_2026-10-24.csv", encoding="utf-8-sig")))
scores = [float(r["Score"] or 0) for r in lb]
band = [s for s in scores if 20 <= s <= 34]
print(f"\nleaderboard 2026-10-24: {len(scores)} teams; top-5 line {sorted(scores)[-5]:.2f}; teams in 20-34: {len(band)}, mean {st.mean(band):.2f}")
dp = fam.get("D-prime slot priority", []) + fam.get("D-prime + patient startup", []) + fam.get("D-prime + patient2 startup", [])
cp = fam.get("Franzen M2 copy", [])
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(band, bins=28, color="#ccc", label=f"teams' best draw in 20-34 (n={len(band)}, mean {st.mean(band):.1f})")
ax2 = ax.twinx(); ax2.set_ylim(0, 3); ax2.set_yticks([])
if cp: ax2.plot(cp, [1] * len(cp), "o", color="#1f4e79", label=f"our copy draws (n={len(cp)}, mean {st.mean(cp):.1f})")
if dp: ax2.plot(dp, [2] * len(dp), "s", color="#c0504d", label=f"our D' draws (n={len(dp)}, mean {st.mean(dp):.1f})")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, frameon=False, fontsize=7, loc="upper left"); ax.set_xlabel("public leaderboard score"); ax.set_ylabel("teams")
ax.set_title("Figure 1 — each team shows its best draw; our per-draw means sit below the band's mean", fontsize=9)
show(fig, "fig1_max_statistic.png")


### 1b. The same reading on ARC-AGI-2 (§2.3, Figure 11)

The sister competition's board is deterministic and quantised (a score is a count of solved tasks), so a copied
notebook is a point mass: the copy band shows as exact-score spikes one task (0.42 points) apart. ARC-AGI-3's board
is stochastic, so the same copy family spreads with sd 2.7–3.5 and the display is a maximum. The noise model's output
for a board is which regime applies and how many draws a difference needs.


In [ ]:
import collections
lb2 = list(csv.DictReader(open(IN / "data/leaderboard_public_arcagi2_2026-10-26.csv", encoding="utf-8-sig")))
nz2 = [float(r["Score"]) for r in lb2 if float(r["Score"] or 0) > 0]; cnt2 = collections.Counter(round(x, 2) for x in nz2)
spikes = sorted([(s, n) for s, n in cnt2.items() if n >= 50], key=lambda t: -t[1])
nz3 = [float(r["Score"]) for r in lb if float(r["Score"] or 0) > 0]; cnt3 = collections.Counter(round(x, 2) for x in nz3)
print(f"ARC-AGI-2: {len(lb2)} teams, {len(nz2)} scoring, {len(cnt2)} distinct scores; teams on spikes of >=50: {sum(n for _, n in spikes)} ({100*sum(n for _, n in spikes)/len(nz2):.0f}%)")
print("  spikes:", spikes[:8])
print(f"ARC-AGI-3: {len(nz3)} scoring, {len(cnt3)} distinct scores; most shared exact scores: {cnt3.most_common(3)}")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].hist([x for x in nz2 if 15 <= x <= 45], bins=300, color="#1f4e79"); axes[0].set_title("ARC-AGI-2: exact-score spikes, one solved task apart", fontsize=9); axes[0].set_ylabel("teams")
axes[1].hist([x for x in nz3 if 15 <= x <= 45], bins=300, color="#c0504d"); axes[1].set_title("ARC-AGI-3: the same copy band, spread; no spikes", fontsize=9)
for a in axes: a.set_xlabel("public score (bins of 0.1)")
show(fig, "fig11_two_regimes.png")


## 2. The allocation simulator (§7.2, Figure 8, §9, Appendix D)

`scripts/sched_sim.py` is an event-driven simulator of the shared-server hidden run, built from 213 real
trajectories (`data/traces.json`: per action, generated tokens and level clears). Cost is generation tokens
divided by 61 tokens/s per stream. It must first reproduce the demonstration runs exactly without contention,
and a contended real run (25 games, 10 slots) to within one action; then, calibrated to the board, it bounds
every slot-priority policy by a clairvoyant oracle.


In [ ]:
spec = importlib.util.spec_from_file_location("ss", IN / "scripts/sched_sim.py"); ss = importlib.util.module_from_spec(spec); spec.loader.exec_module(ss)
ss.TRACES = IN / "data/traces.json"
traces = json.loads(ss.TRACES.read_text()); hp, hg = ss.build_hazards(traces)
print("runs in traces:", {k: len(v) for k, v in traces["runs"].items()})
print("[validate] no contention, no continuation -> the recorded demonstration levels must come back exactly:")
for which in ("copy-franzen-m2", "franzen-rep", "franzen-rep2", "hardtrace"):
    games = ss.run_population(traces, which)
    ss.simulate(games, ss.policy_base, slots=999, wall=10 * ss.WALL_BUDGET_S, allow_continue=False)
    print(f"  {which:>16}: sim {ss.total_levels(games):3d}  trace {sum(g['levels'] for g in traces['runs'][which].values()):3d}")
run = traces["runs"]["hardtrace"]
print(f"[calib] real contended run (25 games, 10 slots, 1500 s): levels {sum(g['levels'] for g in run.values())} actions {sum(len(g['steps']) for g in run.values())}")
for pname in ("base", "dprime", "fair"):
    games = ss.run_population(traces, "hardtrace")
    ss.simulate(games, ss.POLICIES[pname], slots=10, wall=1500.0, cap=1500.0, allow_continue=False, fade=ss.FADE[pname])
    print(f"  sim {pname:>6}: levels {ss.total_levels(games):3d} actions {sum(g.idx for g in games):4d}")


In [ ]:
# Calibrated hidden run (110 games, 9 h, 10 slots): policies across three difficulty regimes -> Figure 8
regimes = [1.0, 1.15, 1.3]; pols = ["fair", "base", "dprime", "oracle"]; SEEDS = range(40, 48)
res = {p: [] for p in pols}
t0 = time.time()
for d in regimes:
    ss.HAZ_SCALE = 0.1; ss.DIFFICULTY = d
    for p in pols:
        ms = []
        for seed in SEEDS:
            games = ss.hidden_population(traces, 110, seed, 0.8)
            ss.simulate(games, ss.POLICIES[p], wall=ss.WALL_BUDGET_S, allow_continue=True, fade=ss.FADE[p])
            ms.append(ss.mean_score(games))
        res[p].append((st.mean(ms), st.pstdev(ms)))
print(f"{len(regimes) * len(pols) * len(SEEDS)} simulated runs in {time.time() - t0:.0f} s")
print(f"{'policy':<8}" + "".join(f"{'difficulty ' + str(d):>18}" for d in regimes))
for p in pols: print(f"{p:<8}" + "".join(f"{m:9.1f} ± {s:4.1f}  " for m, s in res[p]))
fig, ax = plt.subplots(figsize=(6, 3.3)); w = 0.2
colors = {"fair": "#bbb", "base": "#1f4e79", "dprime": "#c0504d", "oracle": "#2e7d32"}
labels = {"fair": "round-robin", "base": "Franzen base", "dprime": "D' (field)", "oracle": "clairvoyant oracle"}
for i, p in enumerate(pols):
    ax.bar([x + (i - 1.5) * w for x in range(3)], [m for m, _ in res[p]], w, yerr=[s for _, s in res[p]], color=colors[p], label=labels[p], capsize=2)
ax.set_xticks(range(3)); ax.set_xticklabels([f"difficulty {d}" for d in regimes]); ax.set_ylabel("simulated contest score")
ax.axhline(28.2, color="#888", ls=":", lw=1); ax.legend(frameon=False, fontsize=8, ncol=2)
ax.set_title("Figure 8 — the oracle sits ~2 above D' and ~4 above base in every regime", fontsize=9)
show(fig, "fig8_allocation_ceiling.png")


In [ ]:
# §9 / Appendix D.7b: pricing cheaper actions. Difficulty 1.10 is today's token cost; every action's tokens are scaled.
factors = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]; pols = ["base", "dprime", "oracle"]; SEEDS = range(40, 50)
print(f"{'token cost':<11}" + "".join(f"{p:>8}" for p in pols))
proj = {}
for f in factors:
    row = []
    for p in pols:
        ss.HAZ_SCALE = 0.1; ss.DIFFICULTY = 1.10 * f; ms = []
        for seed in SEEDS:
            games = ss.hidden_population(traces, 110, seed, 0.8)
            ss.simulate(games, ss.POLICIES[p], wall=ss.WALL_BUDGET_S, allow_continue=True, fade=ss.FADE[p])
            ms.append(ss.mean_score(games))
        row.append(st.mean(ms))
    proj[f] = row; print(f"x{f:<10.1f}" + "".join(f"{v:8.1f}" for v in row))
print("\nrecorded (results/token_cost_projection_2026-10-22.txt):")
print((IN / "results/token_cost_projection_2026-10-22.txt").read_text()[:700])


## 3. Slow, not stuck (§7.1, Figure 7)

Eighteen games the demonstration excludes as "hard", each given a 60-minute slot: cumulative generated tokens
at every level clear. They clear levels — slowly — which is why patience (the field's retune) buys levels and
insight does not.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5)); n = 0; total = 0
for run in ("hardtrace2a", "hardtrace2b"):
    for gid, g in traces["runs"].get(run, {}).items():
        cum = 0; xs = [0]; ys = [0]
        for tok, _w, clr in g["steps"]:
            cum += tok
            if clr: xs.append(cum / 1000); ys.append(ys[-1] + 1)
        xs.append(sum(s[0] for s in g["steps"]) / 1000); ys.append(ys[-1]); total += ys[-1]
        ax.step(xs, ys, where="post", lw=1.2, alpha=0.85); ax.annotate(gid, (xs[-1], ys[-1]), fontsize=7, xytext=(2, 0), textcoords="offset points"); n += 1
ax.axvline(62, color="#c0504d", ls="--", lw=1); ax.text(63, 5.6, "base patience\n(62k tokens)", fontsize=8, color="#c0504d")
ax.axvline(225, color="#2e7d32", ls="--", lw=1); ax.text(226, 5.6, "D' patience\n(225k)", fontsize=8, color="#2e7d32")
ax.set_xlabel("generated tokens in the game (thousands)"); ax.set_ylabel("levels cleared")
ax.set_title(f"Figure 7 — {n} 'hard' games with a 60-minute slot: {total} levels, slow, not stuck", fontsize=9)
show(fig, "fig7_slow_not_stuck.png")


## 4. The 2026-10-24 arms against the base band (§6, Figure 10)

Eleven 25-minute demonstration runs of six interventions on the Milestone-2 chassis (an objective handbook
in the system prompt, its held-out v2, an orienting census, the counting theory of motion as sandbox helpers,
the same forced into every action's result, and the stack), against five base-configuration runs. The dataset
carries each run's `benchmark.json`; the table counts levels cleared and actions per game.


In [ ]:
ARMS = IN / "data/arms_2026-10-24"
def read_run(d):
    runs = json.load(open(d / "benchmark.json"))["game_runs"]
    per = {r["game_id"][:4]: (int(r.get("levels_completed") or 0), len(r.get("history") or [])) for r in runs}
    return per
base_names = ["base-copy-franzen-m2", "base-franzen-rep", "base-franzen-rep2", "dprime-patient", "base-franzen-streams12"]
arm_names = ["handbook", "handbook2", "handbook-v2", "handbook-v2b", "census-p", "motion", "motion2", "motion-push", "motion-push2", "stack", "stack2"]
base = {n: read_run(ARMS / n) for n in base_names if (ARMS / n).exists()}
arms = {n: read_run(ARMS / n) for n in arm_names if (ARMS / n).exists()}
games = sorted(set().union(*[set(p) for p in base.values()]))
rng = {g: (min(p[g][0] for p in base.values()), max(p[g][0] for p in base.values())) for g in games}
bl = [sum(v[0] for v in p.values()) for p in base.values()]
print(f"base band: levels {bl} (mean {st.mean(bl):.1f}, sd {st.pstdev(bl):.1f}); per-game ranges: " + " ".join(f"{g} {a}-{b}" for g, (a, b) in rng.items()))
print(f"\n{'run':<16}{'levels':>7}{'actions':>9}  per game (levels)                       above/below the per-game range")
for n, p in list(base.items()) + list(arms.items()):
    lv = sum(v[0] for v in p.values()); ac = sum(v[1] for v in p.values())
    up = [g for g in games if p.get(g, (0, 0))[0] > rng[g][1]]; dn = [g for g in games if p.get(g, (0, 0))[0] < rng[g][0]]
    print(f"{n:<16}{lv:>7}{ac:>9}  " + " ".join(f"{g}:{p.get(g, (0, 0))[0]}" for g in games) + f"   up {up} below {dn}")
al = [sum(v[0] for v in p.values()) for p in arms.values()]
print(f"\narm runs: levels {al}, mean {st.mean(al):.1f}; base max {max(bl)}, base min {min(bl)} -> above base max: {sum(1 for x in al if x > max(bl))}, below base min: {sum(1 for x in al if x < min(bl))}")
fig, ax = plt.subplots(figsize=(6, 3.4))
for n, p in base.items():
    ax.plot(sum(v[1] for v in p.values()), sum(v[0] for v in p.values()), "o", color="#999", ms=7); ax.annotate(n.replace("base-", ""), (sum(v[1] for v in p.values()), sum(v[0] for v in p.values())), fontsize=7, color="#666", xytext=(4, 3), textcoords="offset points")
ax.axhspan(min(bl), max(bl), color="#ddd", alpha=0.5, lw=0); ax.axhline(st.mean(bl), color="#888", ls=":", lw=1)
cols = ["#1f4e79", "#1f4e79", "#2e7d32", "#2e7d32", "#c0504d", "#8e44ad", "#8e44ad", "#e67e22", "#e67e22", "#16a085", "#16a085"]
for (n, p), c in zip(arms.items(), cols):
    ax.plot(sum(v[1] for v in p.values()), sum(v[0] for v in p.values()), "s", color=c, ms=7); ax.annotate(n, (sum(v[1] for v in p.values()), sum(v[0] for v in p.values())), fontsize=6, xytext=(4, -8), textcoords="offset points")
ax.set_xlabel("actions in the run (10 games)"); ax.set_ylabel("levels cleared")
ax.set_title("Figure 10 (levels vs actions) — every arm run inside the base band", fontsize=9)
show(fig, "fig10_arms_vs_band.png")


## 5. On the real game engine: theory induction, goal census, planner, UNDO (§5.1, §5.2, §5.4, §6)

These three scripts replay the 22 human solutions through the local ARC-AGI-3 engine (`arc-agi` + `arcengine`
from PyPI; the competition's `environment_files/` carry the 25 public games). They need no model.
`test_theory_induction_v3.py` (the version that produced the recorded numbers) and `test_theory_induction.py` (the current
version, whose theory the planner uses) fit the counting theory and predict each human transition before seeing it, and
count which colour the agent enters at every level-completing action (the goal census); the two versions differ in the
rich-theory column only (67% vs 64%), the avatar column is 68% in both; `test_theory_planner.py`
lets a BFS planner in that theory play levels 2+ in the real engine; `test_undo_offline.py` checks whether
UNDO reverts the board. If the engine or the game files are unavailable in this session, the recorded outputs
in part 6 stand in.


In [ ]:
engine_ok = True
try:
    import arc_agi, arcengine  # noqa: F401
except Exception:
    print("installing the engine from PyPI ...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "arc-agi==0.9.9", "arcengine==0.9.3"], capture_output=True, text=True)
    print(r.stdout[-400:], r.stderr[-400:])
    try:
        import arc_agi, arcengine  # noqa: F401
    except Exception as exc:
        engine_ok = False; print("engine unavailable:", exc)
ENV_FILES = COMP / "environment_files"
if not ENV_FILES.exists():
    cands = [p for p in Path("/kaggle/input").rglob("environment_files") if p.is_dir()] if Path("/kaggle/input").exists() else []
    ENV_FILES = cands[0] if cands else ENV_FILES
WORK = OUT / "engine_work"; WORK.mkdir(exist_ok=True)
link = WORK / "environment_files"
if ENV_FILES.exists() and not link.exists():
    import shutil
    shutil.copytree(ENV_FILES, link)  # a COPY, not a symlink: the engine writes metadata.json next to each game, and /kaggle/input is read-only
print("engine:", engine_ok, "| game files:", ENV_FILES.exists(), ENV_FILES, "| games:", len(list(ENV_FILES.iterdir())) if ENV_FILES.exists() else 0)
env = dict(os.environ, HUMAN_REPLAYS=str(IN / "data/official_human_replays"), PYTHONPATH=str(IN / "scripts"), MPLBACKEND="Agg")
def run_script(name, timeout=1500, tail=60):
    t0 = time.time()
    r = subprocess.run([sys.executable, str(IN / "scripts" / name)], cwd=WORK, env=env, capture_output=True, text=True, timeout=timeout)
    out = (r.stdout or "") + ("\n[stderr] " + r.stderr[-800:] if r.returncode else "")
    lines = out.strip().splitlines()
    print(f"$ {name}  ({time.time() - t0:.0f} s, exit {r.returncode})"); print("\n".join(lines[-tail:]))
if engine_ok and ENV_FILES.exists():
    print("(a) the script as recorded on 2026-10-21 (results/theory_induction_v3_2026-10-21.txt): C 68% of 4083, D 67% of 4085")
    run_script("test_theory_induction_v3.py", tail=6)
    print("\n(b) the current script (the theory the planner uses; a later commit added the key-name prior, relocation and a plausibility order):")
    run_script("test_theory_induction.py", tail=6)
else:
    print("skipped: see part 6 for the recorded output")


In [ ]:
if engine_ok and ENV_FILES.exists():
    run_script("test_theory_planner.py", tail=30)
    run_script("test_undo_offline.py", tail=30)
else:
    print("skipped: see part 6 for the recorded output")


## 6. The recorded outputs (Appendix B)

The outputs the paper quotes, as produced in the repository on the dates in their names. Part 5 should
reproduce the theory-induction percentages (72% / 46%), the goal census (76% of completions enter the
game's modal colour), the planner's 0 of 37 levels, and UNDO reverting the board in 1 of 25 games.


In [ ]:
for name in ("theory_induction_v3_2026-10-21.txt", "theory_planner_2026-10-21.txt", "undo_offline_2026-10-06.txt"):
    p = IN / "results" / name
    if p.exists():
        txt = p.read_text(); print(f"===== {name} ({len(txt.splitlines())} lines) =====\n" + "\n".join(txt.splitlines()[-25:]) + "\n")
print("===== scheduler_sim_2026-10-07.md (head) =====\n" + "\n".join((IN / "results/scheduler_sim_2026-10-07.md").read_text().splitlines()[:30]))


In [ ]:
# Keep only our own figures in the saved output: the working copy of the competition's game files is removed.
import shutil
if (OUT / "engine_work").exists():
    shutil.rmtree(OUT / "engine_work", ignore_errors=True)
print("output files:", sorted(p.name for p in OUT.iterdir() if p.is_file()))


## Provenance and licences

- Dataset: `tantan0327/arc3-paper-artefacts` (CC0 for our data and scripts). `data/official_human_replays/` are Tufa Labs'
  assembly of the ARC Prize human study (MIT; see its NOTICE.txt). `data/leaderboard_public_2026-10-24.csv` is the
  public leaderboard as Kaggle serves it.
- Code: MIT-0 / CC0, authored by us; the Kaggle submission notebooks are Daniel Franzen's Milestone-2 notebook
  (Apache-2.0; Tufa Labs' Duck harness, MIT) plus one cell of ours each (`patches/`).
- Paper: `paper.pdf` in the dataset; the repository snapshot lists every file in `SNAPSHOT_MANIFEST.txt`.
